# 05. XGBoost для прогноза TCE

Здесь весь путь от очищенных Parquet до прогноза, интервалов, массы и даты превышения порога. Запускай ячейки сверху вниз.

## 1. Чтение исходных таблиц

In [56]:
from functools import partial
from xgboost import XGBRegressor

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

here = Path.cwd()
ROOT = next(folder for folder in (here, *here.parents) if (folder / "data" / "iterim").is_dir())
DATA = ROOT / "data" / "iterim"
MASS_DATE = pd.Timestamp("2026-01-01")
CELL_SIZE = 50  # м
THRESHOLD = 5.0  # мкг/л
LEVEL = 0.90


def read(name):
    return pd.read_parquet(DATA / f"{name}.parquet")


wells = read("wells")
samples = read("concentration").merge(
    wells[["well_id", "x", "y", "z"]], on="well_id", how="left", validate="many_to_one"
)
queries = read("queries_test")
sources = read("sources").sort_values("source_id")
pumping = read("pumping")
layers = read("cores")[["z_top", "z_bottom", "porosity"]].drop_duplicates()
raster = tifffile.imread(ROOT / "data" / "raw" / "permeability.tif")

START = pd.Timestamp("2020-01-01")
OUTPUT = ROOT / "submission"

## 2. Общая функция признаков

Из координат и даты получаем сезонность, проводимость TIFF, питание, суммарный водоотбор и признаки источников. Один и тот же расчёт подходит для замеров и произвольных точек.

In [57]:
def month_start(dates):
    return dates.dt.to_period("M").dt.to_timestamp()

def make_features(points, raster, zones, sources, recharge, abstraction):
    """Use the same x, y, z, t inputs for training and every kind of prediction."""
    features = points[["x", "y", "z"]].astype(float).reset_index(drop=True)
    date = points["t"].reset_index(drop=True)
    if date.isna().any() or not np.isfinite(features).all().all():
        raise ValueError("Every point needs finite x, y, z and a valid date")

    angle = 2 * np.pi * (date.dt.month - 1) / 12
    features["month_sin"] = np.sin(angle)
    features["month_cos"] = np.cos(angle)
    features["days_since_start"] = (date - START).dt.days

    row = np.floor(features["y"] / CELL_SIZE).to_numpy(dtype=int)
    col = np.floor(features["x"] / CELL_SIZE).to_numpy(dtype=int)
    height, width = raster.shape
    if not ((0 <= row) & (row < height) & (0 <= col) & (col < width)).all():
        raise ValueError("Point outside the permeability raster")
    values = raster[row, col]
    features["permeability"] = np.where(values == -9999, np.nan, values)

    zone_id = pd.Series(pd.NA, index=features.index, dtype="string")
    for zone in zones.itertuples(index=False):
        inside = (
            features["x"].between(zone.x_min, zone.x_max, inclusive="left")
            & features["y"].between(zone.y_min, zone.y_max, inclusive="left")
        )
        zone_id.loc[inside] = zone.zone
    month = month_start(date)
    schedule_keys = pd.MultiIndex.from_arrays([zone_id, month], names=["zone", "t"])
    features["recharge_mm"] = recharge.reindex(schedule_keys).to_numpy(dtype=float)
    features["abstraction_m3_day"] = abstraction.reindex(month).to_numpy(dtype=float)
    if features[["recharge_mm", "abstraction_m3_day"]].isna().any().any():
        raise ValueError("Missing recharge or abstraction schedule for a point")

    for source in sources.itertuples(index=False):
        dx = features["x"] - source.x
        dy = features["y"] - source.y
        dz = features["z"] - (source.z_top + source.z_bottom) / 2
        features[f"distance_{source.source_id}"] = np.sqrt(dx**2 + dy**2 + dz**2)
        active = date.ge(source.start_date)
        if pd.notna(source.end_date):
            active &= date.le(source.end_date)
        features[f"release_{source.source_id}"] = np.where(
            active, source.estimated_release_rate_g_day, 0.0
        )

    return features


build_features = partial(
    make_features,
    raster=raster,
    zones=read("zones"),
    sources=sources,
    recharge=read("recharge").set_index(["zone", "t"])["recharge"],
    abstraction=read("abstraction").set_index("t")["total_abstraction_m3_day"],
)
X = build_features(samples)
y = samples["concentration"]
display(X.head())

,x,y,z,month_sin,month_cos,days_since_start,permeability,recharge_mm,abstraction_m3_day,distance_SRC-01,release_SRC-01,distance_SRC-02,release_SRC-02,distance_SRC-03,release_SRC-03
0,970.08,1422.65,-35.83,0.866025,-5.000000e-01,858,52.973297,11.35,1446.3,424.223901,78.0,367.650293,31.0,754.928962,49.0
1,1491.46,1751.22,-46.12,-0.500000,8.660254e-01,348,53.985538,10.57,1029.2,933.947040,78.0,417.344647,31.0,401.413197,0.0
2,823.59,1326.11,-16.35,-1.000000,-1.836970e-16,1014,39.224617,12.43,1205.5,351.828854,78.0,514.768339,31.0,906.971026,49.0
3,223.30,2572.96,-9.08,1.000000,6.123234e-17,1924,28.338331,20.93,1574.8,1058.119983,78.0,1167.349655,0.0,1890.366953,0.0
4,345.08,1762.65,-6.27,0.500000,8.660254e-01,1872,53.043087,3.27,1093.7,296.798992,78.0,730.513670,0.0,1420.880439,0.0


## 3. Обучение градиентного бустинга

В густо наблюдаемых блоках каждой пробе даём меньший вес. Для отчёта ограничиваем прогноз снизу нулём.

In [58]:
def fit_model(features, target):
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        min_child_weight=4,
        subsample=0.85,
        tree_method="hist",
        random_state=42,
        n_jobs=2,
    )
    # Keep dense clusters from dominating the fit. Counts use training rows only.
    blocks = (features[["x", "y"]] // 400).astype(int)
    count = blocks.groupby(["x", "y"])["x"].transform("size")
    weight = 1 / np.sqrt(count)
    model.fit(features, target, sample_weight=weight / weight.mean())
    return model

def predict(model, features):
    return np.maximum(model.predict(features).astype(float), 0.0)

## 4. Калибровка интервалов и проверка по времени

Учим модель на данных до 2024 года, подбираем ширину интервала по ошибкам 2024 года и проверяем оба результата на 2025 годе. Для итоговой модели сдвигаем обучение на год вперёд и калибруем по 2025 году. Из-за смещения по времени фактическое покрытие может отличаться от 90%.

In [59]:
def calibrate_interval(model, features, target):
    center = predict(model, features)
    errors = np.abs(target.to_numpy(dtype=float) - center)
    scores = errors / np.sqrt(1 + center)
    rank = min(len(scores), int(np.ceil((len(scores) + 1) * LEVEL)))
    return float(np.partition(scores, rank - 1)[rank - 1])


def predict_interval(model, features, factor):
    center = predict(model, features)
    radius = factor * np.sqrt(1 + center)
    return center, np.maximum(center - radius, 0.0), center + radius

In [60]:
dates = samples["t"]
before_2024 = dates.lt("2024-01-01")
year_2024 = dates.dt.year.eq(2024)
year_2025 = dates.dt.year.eq(2025)
before_2025 = dates.lt("2025-01-01")
assert before_2024.any() and year_2024.any() and year_2025.any()

check_model = fit_model(X.loc[before_2024], y.loc[before_2024])
check_factor = calibrate_interval(check_model, X.loc[year_2024], y.loc[year_2024])
center, low, high = predict_interval(check_model, X.loc[year_2025], check_factor)
actual = y.loc[year_2025].to_numpy(dtype=float)

rmse = np.sqrt(np.mean((actual - center) ** 2))
coverage = np.mean((low <= actual) & (actual <= high))
print(f"Проверка на 2025 годе: RMSE {rmse:.3f} мкг/л, покрытие {coverage:.1%}")

# Модель для выдачи прогнозов использует 2024 год в обучении и 2025 в калибровке.
model = fit_model(X.loc[before_2025], y.loc[before_2025])
interval_factor = calibrate_interval(model, X.loc[year_2025], y.loc[year_2025])

Проверка на 2025 годе: RMSE 4.153 мкг/л, покрытие 85.9%


## 5. Прогноз для queries_test

Строим признаки той же функцией, что и для замеров. Сохраняем по одной строке на запрос с исходным query_id.

In [61]:
X_query = build_features(queries)
center, low, high = predict_interval(model, X_query, interval_factor)
predictions = pd.DataFrame({
    "query_id": queries["query_id"],
    "prediction": center,
    "lower_90": low,
    "upper_90": high,
})
assert len(predictions) == len(queries) and predictions["query_id"].is_unique
display(predictions.head())

,query_id,prediction,lower_90,upper_90
0,Q-0341,2.489311,0.150093,4.828529
1,Q-0488,0.927985,0.000000,2.666797
2,Q-0039,2.462503,0.132288,4.792717
3,Q-0524,2.271209,0.006278,4.536140
4,Q-0741,0.000000,0.000000,1.252278


## 6. Масса на 1 января 2026 года

Для каждого слоя прогнозируем концентрацию в центрах ячеек. Умножаем её на объём воды с учётом пористости и суммируем. Это оценка растворённой массы.

In [62]:
def estimate_mass(model, build_features, raster, layers):
    rows, cols = np.indices(raster.shape)
    x = (cols.ravel() + 0.5) * CELL_SIZE
    y = (rows.ravel() + 0.5) * CELL_SIZE

    mass_mg = 0.0
    for layer in layers.itertuples(index=False):
        depth = (layer.z_top + layer.z_bottom) / 2
        points = pd.DataFrame({"x": x, "y": y, "z": depth, "t": MASS_DATE})
        concentration = predict(model, build_features(points))
        water_m3 = CELL_SIZE**2 * (layer.z_top - layer.z_bottom) * layer.porosity
        # 1 µg/L equals 1 mg/m³.
        mass_mg += concentration.sum() * water_m3
    return float(mass_mg / 1_000_000)

In [63]:
mass_kg = estimate_mass(model, build_features, raster, layers)
print(f"Растворённая масса на {MASS_DATE.date()}: {mass_kg:.2f} кг")

Растворённая масса на 2026-01-01: 403.62 кг


## 7. Порог на водозаборе

Строим ежедневный прогноз для трёх скважин. При смешивании вес каждой скважины равен её расходу. Ищем первую дату, когда итоговая концентрация строго выше 5 мкг/л.

In [64]:
def forecast_intake(model, build_features, wells, pumping, end_date):
    intake = wells.loc[wells["role"].eq("intake"), ["well_id", "x", "y", "z"]]
    days = pd.DataFrame({"t": pd.date_range(MASS_DATE, end_date, freq="D")})
    requests = days.merge(intake, how="cross")
    requests["month"] = month_start(requests["t"])
    rates = pumping[["well_id", "t", "pumping_rate"]].rename(columns={"t": "month"})
    requests = requests.merge(rates, on=["well_id", "month"], how="left", validate="many_to_one")
    if intake.empty or requests["pumping_rate"].isna().any():
        raise ValueError("Missing intake wells or monthly pumping schedule")

    requests["mass_mg_day"] = predict(model, build_features(requests)) * requests["pumping_rate"]
    daily = requests.groupby("t", as_index=False)[["mass_mg_day", "pumping_rate"]].sum()
    daily["prediction"] = daily["mass_mg_day"] / daily["pumping_rate"].where(daily["pumping_rate"] > 0)
    return daily[["t", "pumping_rate", "prediction"]]

In [65]:
intake_forecast = forecast_intake(model, build_features, wells, pumping, queries["t"].max())
arrival = intake_forecast.loc[intake_forecast["prediction"] > THRESHOLD, "t"].min()

if pd.isna(arrival):
    print(f"До {intake_forecast['t'].max().date()} превышения не предсказано.")
else:
    print(f"Первое превышение: {arrival.date()}, через {(arrival - MASS_DATE).days} дней.")
display(intake_forecast.head())

До 2028-12-31 превышения не предсказано.


,t,pumping_rate,prediction
0,2026-01-01,1214.5,1.213305
1,2026-01-02,1214.5,1.213305
2,2026-01-03,1214.5,1.213305
3,2026-01-04,1214.5,1.213305
4,2026-01-05,1214.5,1.213305


## 8. Сохранение прогноза и модели

В конце заново читаем сохранённую модель и проверяем, что она воспроизводит прогнозы и интервалы.

In [66]:
OUTPUT.mkdir(parents=True, exist_ok=True)

predictions.to_csv(OUTPUT / "predictions.csv", index=False)
intake_forecast.to_csv(OUTPUT / "intake_forecast.csv", index=False)
summary = {
    "mass_kg": mass_kg,
    "arrival_date": None if pd.isna(arrival) else arrival.date().isoformat(),
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)

model.get_booster().set_attr(interval_factor=str(interval_factor))
model.save_model(OUTPUT / "model.json")

restored = XGBRegressor()
restored.load_model(OUTPUT / "model.json")
restored_factor = float(restored.get_booster().attr("interval_factor"))
np.testing.assert_allclose(
    np.column_stack(predict_interval(restored, X_query, restored_factor)),
    predictions[["prediction", "lower_90", "upper_90"]].to_numpy(),
)
print(f"Файлы сохранены в {OUTPUT}")

Файлы сохранены в D:\WorkSpace\Projects\Task-01-train\submission
